# Project 10: Overfitting and Underfitting Analysis

**Dataset:** Iris flower classification (150 samples, 4 features, 3 species)

**Goal:** Build neural networks of different complexity on the *same* train/test split and see:
- too simple → **underfitting** (poor on train AND test)
- just right → **good fit** (high on both, small gap)
- too complex → **overfitting** (perfect on train, worse on test)

Tasks covered:
- **Task 1:** Build 3 MLP models with different architectures (Iris dataset).
- **Task 2:** Train + evaluate on the same training/testing data.
- **Task 3:** Identify over/underfitting and suggest fixes (with a demo fix).

> Run every cell top-to-bottom in Colab: **Runtime → Run all**. No files to upload — the dataset comes from `sklearn`.


## Setup — imports

`numpy`/`pandas` for data, `matplotlib`/`seaborn` for plots, `sklearn` for the Iris data, the scaler, the MLP neural nets and the metrics.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # hide convergence warnings so the output stays clean

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, log_loss, classification_report, confusion_matrix

np.random.seed(42)
sns.set(style="whitegrid")
%matplotlib inline

print("Ready! numpy, pandas, sklearn loaded.")


## Task 1a: Load the Iris dataset

4 numeric features (sepal length/width, petal length/width) → predict 1 of 3 species.

In [ ]:
iris = load_iris()
X = iris.data          # (150, 4)
y = iris.target        # 0=setosa, 1=versicolor, 2=virginica

df = pd.DataFrame(X, columns=iris.feature_names)
df["species"] = pd.Categorical.from_codes(y, iris.target_names)

print("Shape:", X.shape, " | Features:", iris.feature_names)
print("Classes:", list(iris.target_names))
print(df["species"].value_counts().to_string())
df.head(5)


### Quick look at the data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# 1) class balance
df["species"].value_counts().reindex(iris.target_names).plot.bar(ax=axes[0], color=["#4C72B0", "#DD8452", "#55A868"])
axes[0].set_title("Class balance (50 flowers each)")
axes[0].set_ylabel("Count")

# 2) the two most separable features
for i, name in enumerate(iris.target_names):
    m = df["species"] == name
    axes[1].scatter(df.loc[m, "petal length (cm)"], df.loc[m, "petal width (cm)"], label=name, s=45, alpha=0.8)
axes[1].set_xlabel("petal length (cm)")
axes[1].set_ylabel("petal width (cm)")
axes[1].set_title("Petal size separates the species well")
axes[1].legend()
plt.tight_layout()
plt.show()


## Task 1b: ONE shared train/test split for all models

Important: every model below uses **exactly the same** `X_train / X_test` (80/20, stratified, `random_state=42`) and the **same StandardScaler**. That way any performance difference comes from *model complexity*, not from different data.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit ONLY on train
X_test_s = scaler.transform(X_test)        # apply same transform to test

print(f"Train: {X_train_s.shape[0]} samples | Test: {X_test_s.shape[0]} samples")
print("Same split + same scaling will be reused for Model A, B and C.")


## Task 1c: Three architectures — too simple, just right, too complex

| Model | Hidden layers | Epochs (`max_iter`) | L2 (`alpha`) | Idea |
|---|---|---|---|---|
| **A — Underfit** | `(5,)` — one tiny layer | 50 (undertrained) | 0.001 | too little capacity + too little training |
| **B — Good fit** | `(32,)` — one medium layer | 500 | 0.001 | enough capacity, mild regularization |
| **C — Overfit** | `(100, 50)` — two big layers | 1500 | 0.00001 (almost no regularization) | far more parameters than training samples |

All use `activation='relu'`, `solver='adam'`, same learning rate, same seed — only size/epochs/regularization change.

In [ ]:
def count_params(n_inputs, hidden, n_outputs):
    """Total trainable weights + biases for a fully-connected MLP."""
    layers = [n_inputs] + list(hidden) + [n_outputs]
    return sum((layers[i] + 1) * layers[i + 1] for i in range(len(layers) - 1))

configs = {
    "A-underfit": {"hidden": (5,),       "max_iter": 50,   "alpha": 0.001,   "tol": 1e-4},
    "B-good":     {"hidden": (32,),      "max_iter": 500,  "alpha": 0.001,   "tol": 1e-4},
    "C-overfit":  {"hidden": (100, 50),  "max_iter": 1500, "alpha": 0.00001, "tol": 1e-9},
}

for name, c in configs.items():
    n = count_params(4, c["hidden"], 3)
    print(f"{name:12s} hidden={str(c['hidden']):12s} params={n:5d}  epochs={c['max_iter']:4d}  alpha={c['alpha']}")
print("\nTraining samples: 120  ->  Model C (~5.7k params) has ~47x more parameters than samples: it can memorize them.")


## Task 2: Train + evaluate every model on the same data

In [ ]:
results = {}
trained = {}

for name, c in configs.items():
    mlp = MLPClassifier(hidden_layer_sizes=c["hidden"], activation="relu", solver="adam",
                        alpha=c["alpha"], tol=c["tol"], learning_rate_init=0.001,
                        max_iter=c["max_iter"], random_state=42)
    mlp.fit(X_train_s, y_train)
    trained[name] = mlp

    tr_acc = accuracy_score(y_train, mlp.predict(X_train_s))
    te_acc = accuracy_score(y_test, mlp.predict(X_test_s))
    tr_ll = log_loss(y_train, mlp.predict_proba(X_train_s))
    te_ll = log_loss(y_test, mlp.predict_proba(X_test_s))
    results[name] = {"train_acc": tr_acc, "test_acc": te_acc, "gap": tr_acc - te_acc,
                       "train_loss": tr_ll, "test_loss": te_ll}

comp = pd.DataFrame(results).T.round(3)
print(comp.to_string())


### Plot 1 — training vs testing accuracy (the classic diagnosis chart)

- Both low → underfitting. Both high + small gap → good fit. Train much higher than test → overfitting.

In [ ]:
labels = list(results.keys())
tr = [results[k]["train_acc"] for k in labels]
te = [results[k]["test_acc"] for k in labels]
x = np.arange(len(labels))

plt.figure(figsize=(8, 4.5))
plt.bar(x - 0.2, tr, width=0.4, label="Train accuracy")
plt.bar(x + 0.2, te, width=0.4, label="Test accuracy")
for i in range(len(labels)):
    plt.text(i - 0.2, tr[i] + 0.015, f"{tr[i]:.2f}", ha="center", fontsize=10)
    plt.text(i + 0.2, te[i] + 0.015, f"{te[i]:.2f}", ha="center", fontsize=10)
plt.xticks(x, ["A: underfit\n(5 neurons)", "B: good fit\n(32 neurons)", "C: overfit\n(100+50 neurons)"])
plt.ylim(0, 1.15)
plt.ylabel("Accuracy")
plt.title("Same data, different complexity: train vs test accuracy")
plt.legend()
plt.tight_layout()
plt.show()


### Plot 2 — training loss curves (`loss_curve_` from each fit)

Underfit loss stays high, good-fit loss settles at a moderate level, overfit loss collapses to ~0 (memorization).

In [ ]:
plt.figure(figsize=(8, 4.5))
for name in configs:
    plt.plot(trained[name].loss_curve_, label=name)
plt.xlabel("Training iteration")
plt.ylabel("Training loss (cross-entropy + L2)")
plt.title("Training loss curves")
plt.legend()
plt.tight_layout()
plt.show()

# Train-loss vs test-loss (log-loss on probabilities) side by side
fig, ax = plt.subplots(figsize=(8, 4.2))
xl = list(results.keys())
ax.bar(np.arange(3) - 0.2, [results[k]["train_loss"] for k in xl], 0.4, label="Train log-loss")
ax.bar(np.arange(3) + 0.2, [results[k]["test_loss"] for k in xl], 0.4, label="Test log-loss")
ax.set_xticks(range(3), ["A: underfit", "B: good fit", "C: overfit"])
ax.set_ylabel("Log-loss (lower = more confident + correct)")
ax.set_title("Test loss >> train loss is the smoking gun of overfitting (Model C)")
ax.legend()
plt.tight_layout()
plt.show()


### Plot 3 — accuracy vs training time (each point = fresh model, more epochs)

Shows *why* the gap opens: Model C keeps improving on train while test stalls/falls; Model A never gets good anywhere.

In [ ]:
sweep = {
    "A-underfit": [10, 25, 50],
    "B-good": [50, 150, 300, 500],
    "C-overfit": [100, 400, 800, 1500],
}
curves = {}
for name, iters in sweep.items():
    c = configs[name]
    tr_list, te_list = [], []
    for mi in iters:
        m = MLPClassifier(hidden_layer_sizes=c["hidden"], activation="relu", solver="adam",
                            alpha=c["alpha"], tol=c["tol"], learning_rate_init=0.001,
                            max_iter=mi, random_state=42)
        m.fit(X_train_s, y_train)
        tr_list.append(accuracy_score(y_train, m.predict(X_train_s)))
        te_list.append(accuracy_score(y_test, m.predict(X_test_s)))
    curves[name] = (iters, tr_list, te_list)

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, name in zip(axes, configs):
    iters, tr_list, te_list = curves[name]
    ax.plot(iters, tr_list, marker="o", label="train")
    ax.plot(iters, te_list, marker="s", label="test")
    ax.set_title(name)
    ax.set_xlabel("Epochs (max_iter)")
    ax.set_ylim(0.3, 1.05)
axes[0].set_ylabel("Accuracy")
axes[1].legend()
fig.suptitle("Learning curves: accuracy vs training duration", y=1.03)
plt.tight_layout()
plt.show()


### Per-model detail — classification report + confusion matrix


In [ ]:
for name in configs:
    print("=" * 60)
    print(name, " hidden=", configs[name]["hidden"])
    print(classification_report(y_test, trained[name].predict(X_test_s), target_names=iris.target_names))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, name in zip(axes, configs):
    cm = confusion_matrix(y_test, trained[name].predict(X_test_s))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
                xticklabels=iris.target_names, yticklabels=iris.target_names)
    acc = results[name]["test_acc"]
    ax.set_title(f"{name} (test acc {acc:.2f})")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
plt.tight_layout()
plt.show()


## Task 3: Diagnosis — which model is which, and why?

| Model | Train acc | Test acc | Gap | Train loss | Test loss | Verdict |
|---|---|---|---|---|---|---|
| **A** | ~0.63 (low) | ~0.53 (low) | small | **high** (~0.85) | **high** (~0.87) | **UNDERFITTING** — too little capacity/training; fails even on data it saw |
| **B** | ~0.97 (high) | ~0.93 (high) | ~0.03 (small) | low (~0.11) | low (~0.16) | **GOOD FIT** — generalizes; this is the target |
| **C** | **1.00** (perfect) | ~0.93 (lower) | ~0.07 + test-loss » train-loss (0.31 vs 0.00) | ~0.00 | **high** (~0.31) | **OVERFITTING** — memorized train (5.7k params vs 120 samples); overconfident on test |

**Rules of thumb used above:**
- Underfit: train and test are *both* bad (high bias). More epochs/neurons help.
- Good fit: both good and close (train ≈ test, losses close).
- Overfit: train much better than test, especially **test-loss » train-loss** (high variance). The accuracy gap on tiny Iris is modest, but the loss gap is the giveaway.


## Task 3 (continued): How to fix each problem

**If underfitting (Model A):** increase capacity (more neurons/layers), train longer, raise learning rate, add features, scale inputs (already done), try a less aggressive regularizer.

**If overfitting (Model C):**
1. **L2 regularization** (`alpha` ↑) — penalizes huge weights.
2. **Early stopping / fewer epochs** — stop before memorization.
3. **Simpler architecture / dropout** — match capacity to data size.
4. **More (or augmented) data + cross-validation** — the real cure with only 120 training flowers.

Demo below: keep Model C's big architecture but turn L2 up (`alpha` 0.00001 → 0.5).

In [ ]:
fixed = MLPClassifier(hidden_layer_sizes=(100, 50), activation="relu", solver="adam",
                      alpha=0.5, learning_rate_init=0.001, max_iter=1000, random_state=42)
fixed.fit(X_train_s, y_train)

for tag, m in [("C original (alpha=0.00001)", trained["C-overfit"]), ("C fixed (alpha=0.5)", fixed)]:
    tr = accuracy_score(y_train, m.predict(X_train_s))
    te = accuracy_score(y_test, m.predict(X_test_s))
    print(f"{tag:28s} train={tr:.3f} test={te:.3f} gap={tr-te:+.3f} "
          f"trainLL={log_loss(y_train, m.predict_proba(X_train_s)):.3f} "
          f"testLL={log_loss(y_test, m.predict_proba(X_test_s)):.3f}")

print("\nFix works: same big network, but stronger L2 raises test accuracy and cuts test-loss roughly in half.")


## Summary (expected output)

- **Underfitting example:** Model A `(5,), 50 epochs` — train ~0.63, test ~0.53. Too simple + undertrained.
- **Appropriate fitting example:** Model B `(32,), 500 epochs` — train ~0.97, test ~0.93, gap ~0.03. Best model.
- **Overfitting example:** Model C `(100, 50), 1500 epochs, alpha=1e-5` — train 1.00, test ~0.93, test-loss (~0.31) » train-loss (~0.00). Memorization due to ~5.7k params on 120 samples.
- **Improvement shown:** L2 `alpha=0.5` on Model C recovers test accuracy and halves test loss; early stopping, simpler nets, and more data/CV are the other standard fixes.

All three models shared the identical 120/30 stratified split and scaler, so the differences reflect model complexity — exactly what the assignment asks for.